In [1]:
##### Calculates raster of income percentile within each country (nationally defined)
# based on GDP per capita

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from pathlib import Path
import rasterstats
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.plot import show, plotting_extent
import matplotlib.pyplot as plt
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt


In [2]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent 

# country geographies 
countries = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer="ADM_0")

# reference raster
ref_path = f"{cd}/Data/Clean/Predictors/Rasters/terrain_slope.tif"
ref_path_reprojected = f"{cd}/Results/Raster_model/reprojected/terrain_slope.tif"

# rasters
GDP = f"{cd}/Data/Raw/Predictors/GDP_Kummu/rast_gdpTot_1990_2022_5arcmin.tif"
GDP_adm2 = f"{cd}/Data/Raw/Predictors/GDP_Kummu/rast_adm2_gdp_perCapita_1990_2022.tif"

population = f"{cd}/Data/Raw/Predictors/GHS_population/GHS_POP_E2030_GLOBE_R2023A_54009_100_V1_0.tif"

# save paths
pop_path = f"{cd}/Data/Raw/Predictors/GHS_population/total_pop_2020_resampled.tif"
pop_path_reprojected = f"{cd}/Data/Raw/Predictors/GHS_population/total_pop_2020_resampled_reprojected.tif"

GDP_path = f"{cd}/Data/Raw/Predictors/GDP_Kummu/GDP_2020_resampled.tif"
GDP_path_reprojected = f"{cd}/Data/Raw/Predictors/GDP_Kummu/GDP_2020_resampled_reprojected.tif"

GDP_adm2_path = f"{cd}/Data/Raw/Predictors/GDP_Kummu/GDP_2020_adm2_resampled.tif"
GDP_adm2_path_reprojected = f"{cd}/Data/Raw/Predictors/GDP_Kummu/GDP_2020_adm2_resampled_reprojected.tif"

GDP_pc_path = f"{cd}/Data/Raw/Predictors/GDP_Kummu/GDP_pc_2020_resampled.tif"
GDP_pc_path_reprojected = f"{cd}/Data/Raw/Predictors/GDP_Kummu/GDP_pc_2020_resampled_reprojected.tif"

percentiles = f"{cd}/Data/Clean/GDP_percentiles/GDP_pc_percentiles.tif"
percentiles_reprojected = f"{cd}/Data/Clean/GDP_percentiles/GDP_pc_percentiles_reprojected.tif"

global_percentiles = f"{cd}/Data/Clean/GDP_percentiles/global_GDP_pc_percentiles.tif"
global_percentiles_reprojected = f"{cd}/Data/Clean/GDP_percentiles/global_GDP_pc_percentiles_reprojected.tif"

In [3]:
##### Resample GDP and population (sum) (TAKES LONG TIME TO RUN FOR POPULATION)
# all land pixels with missing population/GDP data get treated as 0 population/GDP
# all non-land pixels with missing population/GDP data stay as missing data
# reprojecting to both origional project resolution and reprojection needed for PA overlay analysis 

def resample_sum_to_ref(in_path, ref_path, out_path, band=1, atol=1e-9, num_threads=8, warp_mem_limit=4096):
    """
    Resample (sum-aggregate) a source raster onto a reference raster's grid/CRS.
    - Missing source data on land pixels -> 0
    - Non-land reference pixels -> NaN
    """

    with rasterio.open(ref_path) as ref:
        dst_crs = ref.crs
        dst_transform = ref.transform
        dst_shape = ref.shape

        ref_masked = ref.read(1, masked=True)
        ref_nodata_mask = np.ma.getmaskarray(ref_masked)

        ref_res = ref.res

    with rasterio.open(in_path) as src:

        src_res = src.res

        # ---------------------------------------------------------
        # Check whether grids already match
        # ---------------------------------------------------------
        grids_match = (
            src.crs == dst_crs
            and src.shape == dst_shape
            and src.transform.almost_equals(dst_transform, precision=atol)
        )

        if grids_match:
            arr = (
                src.read(band, masked=True)
                .filled(np.nan)
                .astype(np.float32)
            )

            out_meta = src.meta.copy()
            out_meta.update(dtype="float32", count=1, nodata=np.nan)

            with rasterio.open(out_path, "w", **out_meta) as dst:
                dst.write(arr, 1)

            print(
                f"{in_path} (band {band}): "
                "grid already matches ref_path — saved without resampling."
            )
            return arr

        # ---------------------------------------------------------
        # Check resolution in common CRS
        # ---------------------------------------------------------
        if src.crs != dst_crs:
            proj_transform, proj_w, proj_h = calculate_default_transform(
                src.crs, dst_crs, src.width, src.height, *src.bounds
            )
            src_res_in_dst_crs = (abs(proj_transform.a), abs(proj_transform.e))
        else:
            src_res_in_dst_crs = src_res

        if not (
            ref_res[0] >= src_res_in_dst_crs[0] - atol
            and ref_res[1] >= src_res_in_dst_crs[1] - atol
        ):
            raise ValueError(
                f"ref_path resolution {ref_res} ({dst_crs}) is not coarser "
                f"than in_path resolution {src_res_in_dst_crs} projected "
                f"into {dst_crs} "
                f"(native: {src_res} in {src.crs})"
            )

        # ---------------------------------------------------------
        # Calculate source total WITHOUT loading entire raster
        # (float32, no masked-array overhead)
        # ---------------------------------------------------------
        src_nodata = src.nodatavals[band - 1]
        src_valid_sum = 0.0

        for _, window in src.block_windows(band):
            data = src.read(band, window=window, masked=False).astype(np.float32)
            if src_nodata is not None:
                valid = data != src_nodata
                src_valid_sum += np.where(valid, data, 0).sum(dtype=np.float64)
            else:
                src_valid_sum += data.sum(dtype=np.float64)

        # ---------------------------------------------------------
        # Reproject + aggregate using SUM (threaded, larger mem budget)
        # ---------------------------------------------------------
        dst_array = np.full(dst_shape, np.nan, dtype=np.float32)

        reproject(
            source=rasterio.band(src, band),
            destination=dst_array,

            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src_nodata,

            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,

            resampling=Resampling.sum,

            warp_mem_limit=warp_mem_limit,  # MB
            num_threads=num_threads,
        )

    # ---------------------------------------------------------
    # Missing source data on land -> 0
    # Non-land reference pixels -> NaN
    # ---------------------------------------------------------
    dst_array = np.where(
        np.isnan(dst_array) & ~ref_nodata_mask,
        0,
        dst_array
    )
    dst_array[ref_nodata_mask] = np.nan

    # ---------------------------------------------------------
    # Save
    # ---------------------------------------------------------
    out_meta = dict(
        driver="GTiff",
        height=dst_shape[0],
        width=dst_shape[1],
        count=1,
        dtype="float32",
        crs=dst_crs,
        transform=dst_transform,
        nodata=np.nan
    )

    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(dst_array, 1)

    # ---------------------------------------------------------
    # Check conservation
    # ---------------------------------------------------------
    total_after = np.nansum(dst_array)

    print(
        f"{in_path} (band {band}): "
        f"before={src_valid_sum:.1f}, "
        f"after={total_after:.1f}, "
        f"ratio={total_after / src_valid_sum:.4f}"
    )

    return dst_array


# pop_org = resample_sum_to_ref(population, ref_path, pop_path)
pop_PA = resample_sum_to_ref(pop_path, ref_path_reprojected, pop_path_reprojected)
GDP_org = resample_sum_to_ref(GDP, ref_path, GDP_path, band=31)
GDP_PA = resample_sum_to_ref(GDP, ref_path_reprojected, GDP_path_reprojected, band=31)

/Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Raw/Predictors/GHS_population/total_pop_2020_resampled.tif (band 1): before=nan, after=8546122752.0, ratio=nan
/Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Raw/Predictors/GDP_Kummu/rast_gdpTot_1990_2022_5arcmin.tif (band 31): grid already matches ref_path — saved without resampling.
/Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Raw/Predictors/GDP_Kummu/rast_gdpTot_1990_2022_5arcmin.tif (band 31): before=nan, after=126790194429952.0, ratio=nan


In [ ]:
# ##### Resample GDP and population (sum) (OLD)
# # all land pixels with missing population/GDP data get treated as 0 population/GDP
# # all non-land pixels with missing population/GDP data stay as missing data
# # reprojecting to both origional project resolution and reprojection needed for PA overlay analysis 

# def resample_sum_to_ref(in_path, ref_path, out_path, band=1, atol=1e-9):

#     with rasterio.open(ref_path) as ref:
#         dst_crs = ref.crs
#         dst_transform = ref.transform
#         dst_shape = ref.shape

#         # Reference raster mask
#         ref_masked = ref.read(1, masked=True)
#         ref_nodata_mask = np.ma.getmaskarray(ref_masked)

#         ref_res = ref.res

#     with rasterio.open(in_path) as src:

#         src_res = src.res

#         # ---------------------------------------------------------
#         # Check whether grids already match
#         # ---------------------------------------------------------
#         grids_match = (
#             src.crs == dst_crs
#             and src.shape == dst_shape
#             and src.transform.almost_equals(
#                 dst_transform,
#                 precision=atol
#             )
#         )

#         if grids_match:

#             arr = (
#                 src.read(band, masked=True)
#                 .filled(np.nan)
#                 .astype(np.float32)
#             )

#             out_meta = src.meta.copy()
#             out_meta.update(
#                 dtype="float32",
#                 count=1,
#                 nodata=np.nan
#             )

#             with rasterio.open(out_path, "w", **out_meta) as dst:
#                 dst.write(arr, 1)

#             print(
#                 f"{in_path} (band {band}): "
#                 "grid already matches ref_path — saved without resampling."
#             )

#             return arr

#         # ---------------------------------------------------------
#         # Check resolution in common CRS
#         # ---------------------------------------------------------
#         if src.crs != dst_crs:

#             proj_transform, proj_w, proj_h = calculate_default_transform(
#                 src.crs,
#                 dst_crs,
#                 src.width,
#                 src.height,
#                 *src.bounds
#             )

#             src_res_in_dst_crs = (
#                 abs(proj_transform.a),
#                 abs(proj_transform.e)
#             )

#         else:
#             src_res_in_dst_crs = src_res

#         if not (
#             ref_res[0] >= src_res_in_dst_crs[0] - atol
#             and
#             ref_res[1] >= src_res_in_dst_crs[1] - atol
#         ):
#             raise ValueError(
#                 f"ref_path resolution {ref_res} ({dst_crs}) is not coarser "
#                 f"than in_path resolution {src_res_in_dst_crs} projected "
#                 f"into {dst_crs} "
#                 f"(native: {src_res} in {src.crs})"
#             )

#         # ---------------------------------------------------------
#         # Calculate source total WITHOUT loading entire raster
#         # ---------------------------------------------------------
#         src_valid_sum = 0.0

#         for _, window in src.block_windows(band):

#             data = src.read(
#                 band,
#                 window=window,
#                 masked=True
#             )

#             src_valid_sum += data.filled(0).astype(np.float64).sum()

#         # ---------------------------------------------------------
#         # Reproject + aggregate using SUM
#         # ---------------------------------------------------------
#         src_nodata = src.nodatavals[band - 1]

#         dst_array = np.full(
#             dst_shape,
#             np.nan,
#             dtype=np.float32
#         )

#         reproject(
#             source=rasterio.band(src, band),
#             destination=dst_array,

#             src_transform=src.transform,
#             src_crs=src.crs,
#             src_nodata=src_nodata,

#             dst_transform=dst_transform,
#             dst_crs=dst_crs,
#             dst_nodata=np.nan,

#             resampling=Resampling.sum,

#             # Keep GDAL's working memory bounded
#             warp_mem_limit=512
#         )

#     # ---------------------------------------------------------
#     # Missing source data on land -> 0
#     # Non-land reference pixels -> NaN
#     # ---------------------------------------------------------
#     dst_array = np.where(
#         np.isnan(dst_array) & ~ref_nodata_mask,
#         0,
#         dst_array
#     )

#     dst_array[ref_nodata_mask] = np.nan

#     # ---------------------------------------------------------
#     # Save
#     # ---------------------------------------------------------
#     out_meta = dict(
#         driver="GTiff",
#         height=dst_shape[0],
#         width=dst_shape[1],
#         count=1,
#         dtype="float32",
#         crs=dst_crs,
#         transform=dst_transform,
#         nodata=np.nan
#     )

#     with rasterio.open(out_path, "w", **out_meta) as dst:
#         dst.write(dst_array, 1)

#     # ---------------------------------------------------------
#     # Check conservation
#     # ---------------------------------------------------------
#     total_after = np.nansum(dst_array)

#     print(
#         f"{in_path} (band {band}): "
#         f"before={src_valid_sum:.1f}, "
#         f"after={total_after:.1f}, "
#         f"ratio={total_after / src_valid_sum:.4f}"
#     )

#     return dst_array

# pop_org = resample_sum_to_ref(population, ref_path, pop_path)
# pop_PA = resample_sum_to_ref(population, ref_path_reprojected, pop_path_reprojected)
# GDP_org = resample_sum_to_ref(GDP, ref_path, GDP_path, band=31)
# GDP_PA = resample_sum_to_ref(GDP, ref_path_reprojected, GDP_path_reprojected, band=31)

In [4]:
##### Resample GDPpc of adm2 region (average)
# all land pixels with missing GDP pc get filled with GDP pc of nearest pixel with data
# reprojecting to both origional project resolution and reprojection needed for PA overlay analysis 

def resample_avg_to_ref(in_path, ref_path, out_path, band=1, atol=1e-9):
    with rasterio.open(ref_path) as ref:
        dst_crs, dst_transform, dst_shape = ref.crs, ref.transform, ref.shape
        ref_masked = ref.read(1, masked=True)
        ref_nodata_mask = np.ma.getmaskarray(ref_masked)
        ref_res = ref.res

    with rasterio.open(in_path) as src:
        src_res = src.res
        grids_match = (
            src.crs == dst_crs
            and src.shape == dst_shape
            and src.transform.almost_equals(dst_transform, precision=atol)
        )

        # --- already on the same grid: just save, no resampling needed ---
        if grids_match:
            arr = src.read(band, masked=True).filled(np.nan).astype(np.float32)
            out_meta = src.meta.copy()
            out_meta.update(dtype="float32", count=1, nodata=np.nan)
            with rasterio.open(out_path, "w", **out_meta) as dst:
                dst.write(arr, 1)
            print(f"{in_path} (band {band}): grid already matches ref_path — saved without resampling.")
            return arr

        # --- otherwise, resolution check before resampling ---
        if not (ref_res[0] >= src_res[0] - atol and ref_res[1] >= src_res[1] - atol):
            raise ValueError(
                f"ref_path resolution {ref_res} is not coarser than in_path "
                f"resolution {src_res} — averaging resample requires ref_path coarser."
            )

        src_nodata = src.nodatavals[band - 1]  # per-band nodata, falls back to src.nodata for most files
        src_valid_mean = src.read(band, masked=True).mean()  # ignores masked/nodata cells

        dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, band),
            destination=dst_array,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src_nodata,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,
            resampling=Resampling.average,
        )

    # --- fill gaps (valid ref land, no source data) with nearest valid pixel ---
    gap_mask = np.isnan(dst_array) & ~ref_nodata_mask
    if gap_mask.any():
        valid_mask = ~np.isnan(dst_array)
        # for each gap cell, find the index of the nearest valid cell
        _, (nearest_i, nearest_j) = distance_transform_edt(
            ~valid_mask, return_distances=True, return_indices=True
        )
        dst_array[gap_mask] = dst_array[nearest_i[gap_mask], nearest_j[gap_mask]]

    dst_array[ref_nodata_mask] = np.nan  # non-land -> stays missing

    out_meta = dict(driver="GTiff", height=dst_shape[0], width=dst_shape[1],
                     count=1, dtype="float32", crs=dst_crs, transform=dst_transform,
                     nodata=np.nan)
    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(dst_array.astype("float32"), 1)

    mean_after = np.nanmean(dst_array)
    print(f"{in_path} (band {band}): before_mean={src_valid_mean:.4f}, after_mean={mean_after:.4f}, "
          f"ratio={mean_after/src_valid_mean:.4f}")

    return dst_array

GDP_adm2_org = resample_avg_to_ref(GDP_adm2, ref_path, GDP_adm2_path, band=31)
GDP_adm2_PA = resample_avg_to_ref(GDP_adm2, ref_path_reprojected, GDP_adm2_path_reprojected, band=31)

/Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Raw/Predictors/GDP_Kummu/rast_adm2_gdp_perCapita_1990_2022.tif (band 31): grid already matches ref_path — saved without resampling.
/Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Raw/Predictors/GDP_Kummu/rast_adm2_gdp_perCapita_1990_2022.tif (band 31): before_mean=29019.5099, after_mean=23652.5449, ratio=0.8151


In [7]:
##### Create percentile raster
# Step 1: calculate GDP pc of each pixel 
# Step 2: for pixels missing GDP or population, fill in with GDP pc from nearest pixelw ithin country 
# Step 3: check calculated GDP pc of pixels against GDP pc of admin region pixel falls within
#         if its more than X times greater than overall region average then it just gets region average 
#         this helps prevent blowout from very small GDP pc
# Step 4: calculate CDF of GDP pc by population (excluding filled in pixels)
# Step 5: assign pixels to country-specific GDP pc percentiles based on CDF of GDP pc in country 

def gdp_pc_percentile_raster(pop_arr, gdp_arr, ref_path, countries_gdf,
                              admin2_gdp_pc_arr=None, cap_multiplier=5.0,
                              out_path=None, gdp_pc_out_path=None):
    with rasterio.open(ref_path) as ref:
        transform, shape, crs = ref.transform, ref.shape, ref.crs

    countries_gdf = countries_gdf.to_crs(crs).reset_index(drop=True)
    countries_gdf["_cid"] = np.arange(1, len(countries_gdf) + 1)

    country_raster = rasterize(
        list(zip(countries_gdf.geometry, countries_gdf["_cid"])),
        out_shape=shape, transform=transform, fill=0, dtype="int32",
    )

    pop = pop_arr.astype(np.float64)
    gdp = gdp_arr.astype(np.float64)

    has_pop      = ~np.isnan(pop) & (pop > 0)
    pop_zero     = ~np.isnan(pop) & (pop == 0)
    valid_pop_px = has_pop | pop_zero
    valid_gdp    = ~np.isnan(gdp) & (gdp > 0)
    base_valid   = has_pop & valid_gdp

    gdp_pc = np.full(shape, np.nan)
    gdp_pc[base_valid] = gdp[base_valid] / pop[base_valid]

    # --- sanity-check own pixel ratio against admin2 gdp_pc ---
    implausible = np.zeros(shape, dtype=bool)
    if admin2_gdp_pc_arr is not None:
        admin2 = admin2_gdp_pc_arr.astype(np.float64)
        has_admin2 = ~np.isnan(admin2) & (admin2 > 0)

        check = base_valid & has_admin2
        ratio_hi = admin2 * cap_multiplier
        ratio_lo = admin2 / cap_multiplier

        implausible = check & ((gdp_pc > ratio_hi) | (gdp_pc < ratio_lo))

        # fall back to the admin2 value for implausible pixels
        gdp_pc[implausible] = admin2[implausible]

        # these pixels no longer have a *reliable own* ratio, so they should
        # NOT drive the CDF -- treat them like borrowed/filled pixels
        base_valid = base_valid & ~implausible

    fill_target = (valid_pop_px & ~base_valid) | implausible

    percentile_raster = np.full(shape, np.nan, dtype=np.float32)

    for cid in np.unique(country_raster):
        if cid == 0:
            continue
        country_mask = country_raster == cid
        rows, cols = np.where(country_mask)
        rmin, rmax, cmin, cmax = rows.min(), rows.max(), cols.min(), cols.max()

        sl = np.s_[rmin:rmax+1, cmin:cmax+1]
        sub_mask     = country_mask[sl]
        sub_gdp_pc   = gdp_pc[sl].copy()
        sub_base     = base_valid[sl] & sub_mask
        sub_fill_tgt = fill_target[sl] & sub_mask
        sub_pop      = pop[sl]

        if not sub_base.any():
            continue

        if sub_fill_tgt.any():
            _, idx = distance_transform_edt(~sub_base, return_indices=True)
            nearest_vals = sub_gdp_pc[tuple(idx)]
            # only borrow from neighbor where we don't already have an
            # admin2 fallback value (implausible pixels keep their admin2 value)
            need_neighbor = sub_fill_tgt & np.isnan(sub_gdp_pc)
            sub_gdp_pc[need_neighbor] = nearest_vals[need_neighbor]

        gdp_pc[sl] = sub_gdp_pc

        w_gdp = sub_gdp_pc[sub_base]
        w_pop = sub_pop[sub_base]

        order = np.argsort(w_gdp)
        sorted_gdp, sorted_pop = w_gdp[order], w_pop[order]

        unique_vals, inverse = np.unique(sorted_gdp, return_inverse=True)
        pop_per_val = np.zeros(len(unique_vals))
        np.add.at(pop_per_val, inverse, sorted_pop)

        cum_pop = np.cumsum(pop_per_val)
        total_pop = cum_pop[-1]
        midpoint_share = (cum_pop - pop_per_val / 2.0) / total_pop
        pct_int = np.clip(np.ceil(midpoint_share * 100), 1, 100)

        assign_mask = sub_mask & (sub_base | (sub_fill_tgt & ~np.isnan(sub_gdp_pc)))
        assigned = np.interp(sub_gdp_pc[assign_mask], unique_vals, pct_int)
        assigned = np.clip(np.round(assigned), 1, 100).astype(np.float32)

        out_sub = percentile_raster[sl]
        out_sub[assign_mask] = assigned
        percentile_raster[sl] = out_sub

    out_meta = dict(driver="GTiff", height=shape[0], width=shape[1], count=1,
                     dtype="float32", crs=crs, transform=transform, nodata=np.nan)

    if out_path:
        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(percentile_raster, 1)

    if gdp_pc_out_path:
        with rasterio.open(gdp_pc_out_path, "w", **out_meta) as dst:
            dst.write(gdp_pc.astype(np.float32), 1)

    return percentile_raster, gdp_pc


with rasterio.open(GDP_adm2_path) as src:
    GDP_adm2_arr = src.read(1)
with rasterio.open(GDP_adm2_path_reprojected) as src:
    GDP_adm2_arr_reproj = src.read(1)

# gdp_pc_pctl, gdp_pc_raster = gdp_pc_percentile_raster(
#     pop_org,
#     GDP_org,
#     ref_path,
#     countries,
#     admin2_gdp_pc_arr=GDP_adm2_arr,
#     cap_multiplier=3.0,
#     out_path=percentiles,
#     gdp_pc_out_path=GDP_pc_path,
# )

gdp_pc_pctl_reproj, gdp_pc_raster_reproj = gdp_pc_percentile_raster(
    pop_PA,
    GDP_PA,
    ref_path_reprojected,
    countries,
    admin2_gdp_pc_arr=GDP_adm2_arr_reproj,
    cap_multiplier=3.0,
    out_path=percentiles_reprojected,
    gdp_pc_out_path=GDP_pc_path_reprojected,
)

In [8]:
##### Create GLOBAL percentile raster
# Same logic as the country-based version, but ranks pixels against the entire
# global population instead of within-country only. Reuses the gap-filled
# gdp_pc rasters already computed above (so no need to re-run the infilling step).

def gdp_pc_percentile_raster_global(gdp_pc_arr, pop_arr, ref_path, out_path=None):
    with rasterio.open(ref_path) as ref:
        transform, shape, crs = ref.transform, ref.shape, ref.crs

    gdp_pc = gdp_pc_arr.astype(np.float64)
    pop    = pop_arr.astype(np.float64)

    valid = ~np.isnan(gdp_pc) & ~np.isnan(pop) & (pop > 0) & (gdp_pc > 0)

    w_gdp = gdp_pc[valid]
    w_pop = pop[valid]

    order = np.argsort(w_gdp)
    sorted_gdp, sorted_pop = w_gdp[order], w_pop[order]

    # population-weighted CDF over ALL valid pixels globally
    unique_vals, inverse = np.unique(sorted_gdp, return_inverse=True)
    pop_per_val = np.zeros(len(unique_vals))
    np.add.at(pop_per_val, inverse, sorted_pop)

    cum_pop = np.cumsum(pop_per_val)
    total_pop = cum_pop[-1]
    midpoint_share = (cum_pop - pop_per_val / 2.0) / total_pop
    pct_int = np.clip(np.ceil(midpoint_share * 100), 1, 100)

    assigned = np.interp(gdp_pc[valid], unique_vals, pct_int)
    assigned = np.clip(np.round(assigned), 1, 100).astype(np.float32)

    percentile_raster = np.full(shape, np.nan, dtype=np.float32)
    percentile_raster[valid] = assigned

    if out_path:
        out_meta = dict(driver="GTiff", height=shape[0], width=shape[1], count=1,
                         dtype="float32", crs=crs, transform=transform, nodata=np.nan)
        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(percentile_raster, 1)

    return percentile_raster


# global_gdp_pc_pctl = gdp_pc_percentile_raster_global(
#     gdp_pc_raster,
#     pop_org,
#     ref_path,
#     out_path=global_percentiles,
# )

global_gdp_pc_pctl_reproj = gdp_pc_percentile_raster_global(
    gdp_pc_raster_reproj,
    pop_PA,
    ref_path_reprojected,
    out_path=global_percentiles_reprojected,
)